### RAG pipeline - Data ingestion to Vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\Projects\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

def process_pdf(pdf_directory):
    documents = []
    pdf_dir = Path(pdf_directory)
    
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print("no of pdfs= ",len(pdf_files))
    for pdf in pdf_files:
        print("processing", pdf.name)
        try:
            loader = PyPDFLoader(str(pdf))
            document = loader.load()
            
            for doc in document:
                doc.metadata['source_file']=pdf.name
                doc.metadata['file_type']='pdf'
                
            documents.extend(document)
            
            print(f"{len(documents)} documents loaded.")
            
        except Exception as e:
            print(f"error : {e}")
            
    return documents

document = process_pdf("../data")

no of pdfs=  1
processing sorting.pdf
66 documents loaded.


In [3]:
# textsplitting

def splitdocs(documents,chunksize=1000,chunkoverlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunksize,
        chunk_overlap = chunkoverlap,
        length_function = len,
        separators=["\n\n","\n"," ",""])
    
    split_docs=text_splitter.split_documents(documents=documents)
    print(f"split {len(documents)} into {len(split_docs)} chunks")
    
    if split_docs:
        print("\nExample chunk:")
        print(f"Page content: {split_docs[0].page_content[:200]}...")
        print(f"metadata: {split_docs[0].metadata}")
        
    return split_docs

split_docs = splitdocs(document)
    

split 66 into 164 chunks

Example chunk:
Page content: Introduction
This part presents several algorithms that solve the following sorting problem:
Input: A sequence of n numbersha1;a 2;:::;a ni.
Output: A permutation (reordering)ha0
1;a 0
2;:::;a 0
ni of...
metadata: {'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2026-04-18T20:19:22+00:00', 'source': '..\\data\\pdf\\sorting.pdf', 'total_pages': 66, 'page': 0, 'page_label': '1', 'source_file': 'sorting.pdf', 'file_type': 'pdf'}


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
class EmbeddingManager:
    '''Handles document embedding generation using SentenceTransformers'''
    def __init__(self, model_name:str = "sentence-transformers/all-MiniLM-L6-v2"):
        '''
        Uses a hugging face model for sentence transformer.
        '''
        self.model = None
        self.model_name = model_name
        self._load_model()
        
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Embedding model loaded with dimensions {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error while loading SentenceTransformer model {e}")
            raise
        
    def generate_embedding(self,texts:List[str])->np.ndarray:
        ''''
        Generates embeddings for a list of texts.
        Args:
            texts = list of texts which are to be embed.
        Returns:
            numpy array of shape (len(texts), embedding dimension)
        '''
        if not self.model:
            return ValueError("Model not found")
        
        print(f"Generating vectors for {len(texts)} texts:")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings shape - {embeddings.shape}")
        return embeddings

In [6]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4171.70it/s]


Embedding model loaded with dimensions 384


In [7]:
class VectorStore:
    '''
    stores document embeddings to chromadb vector store
    '''
    def __init__(self,collection_name:str='pdf_documents',persist_directory:str = '../data/vector_store'):
        ''' 
        Initializes vector store
        Args:
            collection_name = name of the chromadb collection name
            persist_dictionary = Directory to persist the vector store
        '''
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        ''' 
        Initialize chromadb client and connection
        '''
        try:
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                self.collection_name,
                metadata={"Description":"PDF document embedding for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error while initializing vectore store {e}")
            raise
        
    def add_documents(self,docs: List[Any], embeddings: np.ndarray):
        ''' 
        Adds list of documents to collection store.
        Args:
            doc = list of documents to add
            embeddings = corresponding embeddings for the document
        '''
        if len(docs)!=len(embeddings):
            raise ValueError("Length of document and embeddings must match")
        
        print(f"Adding {len(docs)} documents to vector store")
        
        ids = []
        metadatas = []
        embeddings_list = []
        documents_text = []
        
        for i, (doc,embedding) in enumerate(zip(docs,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())      
            
        try:
            self.collection.add(
                    ids = ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )    
            print(f"Successfully added {len(docs)} to vector store.")
            print(f"Total document in collection: {self.collection.count()}")
                
        except Exception as e:
            print(f"Error while adding documents: {e}")  
            raise

In [8]:
vectorstore = VectorStore()
vectorstore

Vector Store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [9]:
## Converting the texts to embeddings:
texts = [docs.page_content for docs in split_docs]

embeddings=embedding_manager.generate_embedding(texts)

##Store in vector store:
vectorstore.add_documents(split_docs,embeddings)

Generating vectors for 164 texts:


Batches: 100%|██████████| 6/6 [00:04<00:00,  1.33it/s]


Generated embeddings shape - (164, 384)
Adding 164 documents to vector store
Successfully added 164 to vector store.
Total document in collection: 164


### Retrieval Pipeline from the Vector Store

In [10]:
class RAGReitriever:
    ''' 
    Handles query based retrieval from the vector store
    '''
    def __init__(self,vectorstore: VectorStore, embedding_manager: EmbeddingManager):
        ''' 
        Initializes the retriever
        Args:
            vectorstore = the vector which contains document embeddings
            embedding_manager = Manager for generating query embeddings
        '''
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager
        
        
    def retrieve(self, query:str, top_k:int = 5, score_threshold:float = 0.0)-> List[Dict[str,Any]]:
        ''' 
        Retrieve relevant documents for a query
        Args:
            query: query input by user
            top_k: number of top results to return
            score_threshold: minimum score of similary
        Returns:
            List of dictionaries containing retrieved docs and metadata
        '''
        print(f"Retrieving for query {query}")
        print(f"Top K: {top_k} and Score Threshold: {score_threshold}")
        
        query_embeddings = embedding_manager.generate_embedding([query])[0]
        
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embeddings.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
            
rag_retriever = RAGReitriever(vectorstore,embedding_manager)

In [11]:
rag_retriever.retrieve("What is non-comparison sort")

Retrieving for query What is non-comparison sort
Top K: 5 and Score Threshold: 0.0
Generating vectors for 1 texts:


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.80it/s]

Generated embeddings shape - (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5f12fb04_105',
  'content': '8 Sorting in Linear Time\nWe have now introduced several algorithms that can sort n numbers in O.n lg n/\ntime. Merge sort and heapsort achieve this upper bound in the worst case; quicksort\nachieves it on average. Moreover, for each of these algorithms, we can produce a\nsequence of n input numbers that causes the algorithm to run in /DEL.nlg n/ time.\nThese algorithms share an interesting property: the sorted order they determine\nis based only on comparisons between the input elements . We call such sorting\nalgorithms comparison sorts . All the sorting algorithms introduced thus far are\ncomparison sorts.\nIn Section 8.1, we shall prove that any comparison sort must make /DEL.nlg n/\ncomparisons in the worst case to sort n elements. Thus, merge sort and heapsort\nare asymptotically optimal, and no comparison sort exists that is faster by more\nthan a constant factor.\nSections 8.2, 8.3, and 8.4 examine three sorting algorithms—counting sort

## VectorDB context pipeline with LLM output

In [14]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

grok_api_key = os.getenv('grok_api_key')

llm=ChatGroq(groq_api_key=grok_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [15]:
answer=rag_simple("What is non-comparison sort?",rag_retriever,llm)
print(answer)

Retrieving for query What is non-comparison sort?
Top K: 3 and Score Threshold: 0.0
Generating vectors for 1 texts:


Batches: 100%|██████████| 1/1 [00:00<00:00, 101.61it/s]

Generated embeddings shape - (1, 384)
Retrieved 3 documents (after filtering)


Non-comparison sort is a sorting algorithm that does not rely solely on comparisons between elements to determine the sorted order. Instead, it uses operations other than comparisons, such as counting, radix sort, or bucket sort, to achieve linear time complexity.


In [16]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is count sort.", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving for query what is count sort.
Top K: 3 and Score Threshold: 0.1
Generating vectors for 1 texts:


Batches: 100%|██████████| 1/1 [00:00<00:00, 92.00it/s]

Generated embeddings shape - (1, 384)
Retrieved 3 documents (after filtering)


Answer: Counting sort is a sorting algorithm that assumes the input is an array of integers within a specific range (0 to k). It uses two auxiliary arrays, B and C, to sort the input array A in linear time (O(n + k)).
Sources: [{'source': 'sorting.pdf', 'page': 47, 'score': 0.35614585876464844, 'preview': 'same position.\nIn the code for counting sort, we assume that the input is an array AŒ1 : : n/c141,a n d\nthus A:lengthD n. We require two other arrays: the array BŒ 1::n /c141holds the\nsorted output, and the array CŒ 0::k/c141provides temporary working storage....'}, {'source': 'sorting.pdf', 'page': 2, 'score': 0.2552129030227661, 'preview': 'Thus, when kD O.n/, counting sort runs in time that is linear in the size of the\ninput array. A related algorithm, radix sort, can be used to extend the range of\ncounting sort. If there are n integers to sort, each integer has d digits, and each\ndigit can take on up to k possible values, then radix ...'}, {'source': 'sorting.pdf', 'page': 

In [17]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is count sort", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving for query what is count sort
Top K: 3 and Score Threshold: 0.1
Generating vectors for 1 texts:


Batches: 100%|██████████| 1/1 [00:00<00:00, 93.02it/s]

Generated embeddings shape - (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
same position.
In the code for counting sort, we assume that the input is an array AŒ1 : : n/c141,a n d
thus A:lengthD n. We require two other arrays: the array BŒ 1::n /c141holds the
sorted output, and the array CŒ 0::k/c141provides temporary working

 storage.

Thus, when kD O.n/, counting sort runs in time that is linear in the size of the
input array. A related algorithm, radix sort, can be used to extend the range of
counting sort. If there are n integers to sort, each integer has d digits, and each
digit can take on up to k possible values, then radix sort can sort the numbers
in ‚.d.n C k// time. When d is a constant and k is O.n/, radix sort runs in
linear time. A third algorithm, bucket sort, requires knowledge of the probabilistic
distribution of numbers in the input array. It can sort n real numbers uniformly
distributed in the half-open interval Œ0; 1/in average-case O.n/ time.
The following table summarizes the running times of the sorting algorithms from
Chapters 2 and 6–8. As usual,n denotes the number of items to sort. For counting
sort, the items to sort are integers in the setf0; 1; : : : ; kg. For radix sort, each item
is a d-digit number, where each digit takes on k possible values. For bucket sort,

sort used as 